# exp_2b — Experiment 2B: MLP for Iris Classification

Source-based notebook converted from the uploaded experiment code document.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import EarlyStopping

import keras_tuner as kt

In [ ]:
path = r"D:\DL DATASET\Iris.csv"

df = pd.read_csv(path)

print(df.head())

In [ ]:
# Remove ID column
df = df.drop("Id", axis=1)

# Features and Target
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Label Encoding
encoder = LabelEncoder()
y = encoder.fit_transform(y)

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Feature Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training Shape :", X_train.shape)
print("Testing Shape :", X_test.shape)

In [ ]:
def build_model(hp):

    model = Sequential()

    model.add(
        Dense(
            units=hp.Int(
                "units",
                min_value=16,
                max_value=64,
                step=16
            ),
            activation="relu",
            
        )
    )

    model.add(
        Dropout(
            hp.Float(
                "dropout",
                0.0,
                0.3,
                step=0.1
            )
        )
    )

    for i in range(hp.Int("hidden_layers", 1, 3)):
        model.add(
            Dense(
                units=hp.Int(
                    f"units_{i}",
                    min_value=16,
                    max_value=64,
                    step=16
                ),
                activation="relu"
            )
        )

    model.add(Dense(3, activation="softmax"))

    optimizer = hp.Choice(
        "optimizer",
        ["adam", "sgd"]
    )

    learning_rate = hp.Choice(
        "learning_rate",
        [0.01, 0.001, 0.0001]
    )

    if optimizer == "adam":
        opt = Adam(learning_rate=learning_rate)
    else:
        opt = SGD(learning_rate=learning_rate)

    model.compile(
        optimizer=opt,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    overwrite=True,
    directory="Classification_Tuning",
    project_name="IrisClassification"
)

tuner.search(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=16,
    verbose=1
)

In [ ]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best Hyperparameters\n")

print("Units           :", best_hp.get("units"))
print("Dropout         :", best_hp.get("dropout"))
print("Hidden Layers   :", best_hp.get("hidden_layers"))
print("Optimizer       :", best_hp.get("optimizer"))
print("Learning Rate   :", best_hp.get("learning_rate"))

if best_hp.get("hidden_layers") >= 1:
    print("Units Layer 1   :", best_hp.get("units_0"))

if best_hp.get("hidden_layers") == 2:
    print("Units Layer 2   :", best_hp.get("units_1"))

In [ ]:
# ==========================================================
# Train Best Model
# ==========================================================

best_model = tuner.get_best_models(1)[0]

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = best_model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=20,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# ==========================================================
# Evaluate Model
# ==========================================================

loss, accuracy = best_model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("\nTest Loss :", loss)
print("Test Accuracy :", accuracy)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    history.history["accuracy"],
    marker="o",
    linewidth=2,
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    marker="s",
    linewidth=2,
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    history.history["loss"],
    marker="o",
    linewidth=2,
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    marker="s",
    linewidth=2,
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
predictions = best_model.predict(X_test)

predicted_classes = np.argmax(predictions, axis=1)

print("Predicted Classes:")
print(predicted_classes)

print("\nActual Classes:")
print(y_test)